# Task 3: Event Impact Modeling

In [1]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from src import data_loader, impact_model as im, impact_matrix as imx

data = data_loader.load_all()
main_df = data["main"]
impact_df = data["impact"]
events_df = main_df[main_df["record_type"] == "event"].copy()

Loaded 'ethiopia_fi_unified_data': 43 rows, 34 columns
Loaded 'Impact_sheet': 14 rows, 35 columns
Loaded reference codes: 71 rows, 4 columns


## 1. Understand the Impact Data

In [2]:
summary = impact_df.merge(
    events_df[["record_id", "indicator", "category", "observation_date"]],
    left_on="parent_id", right_on="record_id", suffixes=("", "_event")
)
summary[["indicator", "related_indicator", "impact_direction", "impact_magnitude", "lag_months", "evidence_basis"]]

,indicator,related_indicator,impact_direction,impact_magnitude,lag_months,evidence_basis
0,Telebirr effect on Account Ownership,ACC_OWNERSHIP,increase,high,12,literature
1,Telebirr effect on Telebirr Users,USG_TELEBIRR_USERS,increase,high,3,empirical
2,Telebirr effect on P2P Transactions,USG_P2P_COUNT,increase,high,6,empirical
3,Safaricom effect on 4G Coverage,ACC_4G_COV,increase,medium,12,empirical
4,Safaricom effect on Data Affordability,AFF_DATA_INCOME,decrease,medium,12,literature
5,M-Pesa effect on M-Pesa Users,USG_MPESA_USERS,increase,high,3,empirical
6,M-Pesa effect on Mobile Money Account Rate,ACC_MM_ACCOUNT,increase,medium,6,theoretical
7,Fayda effect on Account Ownership,ACC_OWNERSHIP,increase,medium,24,literature
8,Fayda effect on Gender Gap,GEN_GAP_ACC,decrease,medium,24,literature
9,FX Reform effect on Data Affordability,AFF_DATA_INCOME,increase,high,3,empirical


## 2-4. Event-Indicator Association Matrix

In [3]:
matrix = imx.build_association_matrix(impact_df)
matrix.round(2)

related_indicator,ACC_4G_COV,ACC_MM_ACCOUNT,ACC_OWNERSHIP,AFF_DATA_INCOME,GEN_GAP_ACC,USG_MPESA_ACTIVE,USG_MPESA_USERS,USG_P2P_COUNT,USG_TELEBIRR_USERS
parent_id,,,,,,,,,
EVT_0001,NaN,NaN,3.0,NaN,NaN,NaN,NaN,3.0,3.0
EVT_0002,1.5,NaN,NaN,-1.5,NaN,NaN,NaN,NaN,NaN
EVT_0003,NaN,6.33,NaN,NaN,NaN,NaN,3.0,NaN,NaN
EVT_0004,NaN,NaN,1.5,NaN,-1.5,NaN,NaN,NaN,NaN
EVT_0005,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN
EVT_0007,NaN,NaN,NaN,NaN,NaN,1.5,NaN,1.5,NaN
EVT_0008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,NaN
EVT_0010,NaN,NaN,NaN,0.5,NaN,NaN,NaN,NaN,NaN


In [4]:
flags = imx.build_confidence_flags(impact_df)
flags

,indicator,calibrated,calibration_factor
0,ACC_OWNERSHIP,False,1.00
1,USG_TELEBIRR_USERS,False,1.00
2,USG_P2P_COUNT,False,1.00
3,ACC_4G_COV,False,1.00
4,AFF_DATA_INCOME,False,1.00
5,USG_MPESA_USERS,False,1.00
6,ACC_MM_ACCOUNT,True,4.22
7,GEN_GAP_ACC,False,1.00
8,USG_MPESA_ACTIVE,False,1.00


## 5. Validation Against Historical Data
Telebirr (2021) + M-Pesa (2023) predicted effect on Mobile Money Account Rate,
validated against the real observed change (4.7% -> 9.45%, 2021-2024).

In [5]:
predicted, factor, breakdown = im.predict_indicator_change(
    pd.Timestamp("2024-11-29"), "ACC_MM_ACCOUNT", impact_df, events_df)

print(f"Predicted change (calibrated): {predicted:.2f} pp")
print(f"Calibration factor applied: {factor}")
print(f"Real observed change: +4.75 pp (4.7% -> 9.45%)")
print(f"Event breakdown: {breakdown}")

Predicted change (calibrated): 4.75 pp
Calibration factor applied: 4.22
Real observed change: +4.75 pp (4.7% -> 9.45%)
Event breakdown: [{'event_id': 'EVT_0003', 'effect': 1.125}]


**Result:** After calibration, the model's predicted change (4.75pp) matches
the observed change exactly, by construction - calibration was derived from
this same data point. This is a validation of the modeling FRAMEWORK (ramp
function, event summation), not independent proof of accuracy, since no
second held-out data point exists to test against. This limitation is
documented in `docs/impact_methodology.md`.

## 6-7. Methodology, Refinement, and Limitations
See `docs/impact_methodology.md` for full documentation of functional form,
calibration reasoning, comparable-country evidence, assumptions, and limitations.